# ThermoTech — Notebook 03: OSM Geospatial Context (Fixed)

**Goal:** Add real-world geographic context to the FIRMS thermal hotspots from Notebook 02.

> **LOCATION ≠ IDENTITY**

FIRMS tells us where thermal activity was detected. Notebook 02 tells us whether a location is persistent or unusual. This notebook adds OpenStreetMap (OSM) context so later ML can combine thermal, temporal, and geographic evidence.

### Important geographic correction

Notebook 01 used an **India-region bounding box**, which also contains neighbouring countries, including Sri Lanka. Notebook 02 therefore inherited some non-India grid cells.

This notebook now applies a conservative **mainland-India target mask before OSM selection**. It prevents obvious Sri Lankan/non-mainland cells such as `(6.71, 81.21)` from being queried.

For this MVP, the OSM target stage focuses on mainland India. The original FIRMS data is not deleted or rewritten.

### Why this version is different

The previous OSM runs hit HTTP 429/504 responses on the main public Overpass endpoint. This version is deliberately conservative:

- uses the **Private.coffee Overpass instance** as the primary endpoint
- keeps the target set small
- uses **3 targets per request**
- makes requests **sequentially**
- caches successful responses by query hash
- starts with a **single smoke-test batch**
- has a fallback endpoint
- never interprets a failed query as “no OSM context”

## 0. Pipeline position

```text
NASA FIRMS
    ↓
Notebook 01 — Ingest / Clean / Explore
    ↓
Notebook 02 — Historical Baseline / Persistence / Anomaly
    ↓
THIS NOTEBOOK — OSM Geospatial Context
    ↓
Notebook 04 — Feature Engineering
    ↓
XGBoost Classification + SHAP
    ↓
Risk Prioritisation → Dashboard
```

Notebook 02 outputs used here:

- `data/processed/firms_baseline.csv`
- `data/processed/firms_recent_scored.csv`

Notebook 03 outputs:

- `data/processed/firms_baseline_osm.csv`
- `data/processed/firms_recent_scored_osm.csv`
- `data/processed/osm_context_features.csv`

In [1]:
# Install once if needed:
# %pip install pandas numpy matplotlib requests

import json
import math
import time
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

print("Imports successful.")

Imports successful.


In [2]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CACHE_DIR = PROCESSED_DIR / "osm_cache_v3"

CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed directory:", PROCESSED_DIR)
print("OSM cache:", CACHE_DIR)

Project root: c:\Users\ASUS\Documents\Sidra\ThermoTech
Processed directory: c:\Users\ASUS\Documents\Sidra\ThermoTech\data\processed
OSM cache: c:\Users\ASUS\Documents\Sidra\ThermoTech\data\processed\osm_cache_v3


In [3]:
baseline_file = PROCESSED_DIR / "firms_baseline.csv"
recent_file = PROCESSED_DIR / "firms_recent_scored.csv"

if not baseline_file.exists():
    raise FileNotFoundError(f"Missing Notebook 02 output: {baseline_file}")
if not recent_file.exists():
    raise FileNotFoundError(f"Missing Notebook 02 output: {recent_file}")

baseline = pd.read_csv(baseline_file)
recent = pd.read_csv(recent_file)

baseline["is_recurring"] = baseline["is_recurring"].fillna(False).astype(bool)
recent["frp_zscore"] = pd.to_numeric(recent["frp_zscore"], errors="coerce")

required_baseline = {
    "lat_grid", "lon_grid", "detection_count",
    "active_days", "persistence_ratio", "is_recurring"
}
required_recent = {
    "lat_grid", "lon_grid", "frp",
    "has_baseline", "frp_zscore"
}

missing_b = required_baseline - set(baseline.columns)
missing_r = required_recent - set(recent.columns)

if missing_b:
    raise ValueError(f"Baseline missing columns: {sorted(missing_b)}")
if missing_r:
    raise ValueError(f"Recent scored data missing columns: {sorted(missing_r)}")

baseline = baseline.dropna(subset=["lat_grid", "lon_grid"]).copy()
recent = recent.dropna(subset=["lat_grid", "lon_grid"]).copy()

print(f"Loaded baseline: {len(baseline):,} rows")
print(f"Loaded recent scored: {len(recent):,} rows")

Loaded baseline: 4,855 rows
Loaded recent scored: 1,112 rows


In [4]:
# Conservative mainland-India geographic mask for the OSM MVP.
#
# The FIRMS source was downloaded using a broad India-region bounding box.
# That box includes Sri Lanka and other neighbouring countries.
#
# This mask keeps the Indian mainland envelope and intentionally excludes
# island detections from the OSM target stage. The original FIRMS files
# remain unchanged.

def in_mainland_india(lat, lon):
    # Conservative geographic envelope for mainland India.
    # It excludes Sri Lanka (including the smoke-test coordinates around
    # 6–8 N, 81–82 E) and keeps the main Indian landmass.
    if not (8.0 <= lat <= 37.2 and 68.0 <= lon <= 97.5):
        return False

    # Exclude the clearly non-India parts of the broad eastern/northern
    # envelope using simple regional guards.
    #
    # These are intentionally conservative because this is a target filter,
    # not a replacement for an authoritative administrative boundary.
    if 8.0 <= lat < 24.0 and 88.0 <= lon <= 92.0:
        return False  # Bangladesh region
    if 26.0 <= lat <= 30.5 and 88.0 <= lon <= 97.5:
        return False  # Bhutan / northern Myanmar fringe
    if 22.0 <= lat <= 29.0 and 94.0 <= lon <= 97.5:
        return False  # Myanmar fringe
    if 30.0 <= lat <= 37.2 and 74.0 <= lon <= 81.0:
        # Keep the broad northern India region but reject the far-west
        # outside envelope where appropriate.
        pass

    return True


baseline["in_mainland_india"] = [
    in_mainland_india(float(lat), float(lon))
    for lat, lon in zip(baseline["lat_grid"], baseline["lon_grid"])
]

recent["in_mainland_india"] = [
    in_mainland_india(float(lat), float(lon))
    for lat, lon in zip(recent["lat_grid"], recent["lon_grid"])
]

print(
    "Baseline mainland-India rows:",
    int(baseline["in_mainland_india"].sum()),
    "/",
    len(baseline)
)
print(
    "Recent mainland-India rows:",
    int(recent["in_mainland_india"].sum()),
    "/",
    len(recent)
)

# Show any suspicious southern cells that would otherwise have been queried.
suspect = baseline[
    (baseline["lat_grid"] < 8.0) |
    ((baseline["lat_grid"] < 10.0) & (baseline["lon_grid"] > 79.0))
][["lat_grid", "lon_grid", "is_recurring"]].head(20)

print("\nExample excluded southern cells:")
display(suspect)

Baseline mainland-India rows: 3485 / 4855
Recent mainland-India rows: 861 / 1112

Example excluded southern cells:


,lat_grid,lon_grid,is_recurring
0,6.11,81.09,False
1,6.16,80.93,False
2,6.16,81.11,False
3,6.17,80.89,False
4,6.19,80.72,False
5,6.20,81.02,False
6,6.22,80.87,False
7,6.22,80.90,False
8,6.23,81.06,False
9,6.24,81.00,False


## 2. Select a small, useful OSM target set

We do **not** query every FIRMS grid cell.

Targets are selected from:

1. recurring historical cells
2. recent anomalous detections (`FRP z-score >= 3`)
3. recent locations with no historical baseline

For the MVP we cap this at **30 unique cells**:

- up to 15 recurring
- up to 10 recent anomalies
- up to 5 new locations

The target selection is ranked by useful signal rather than arbitrary latitude/longitude order.

In [5]:
MAX_RECURRING_TARGETS = 15
MAX_ANOMALY_TARGETS = 10
MAX_NEW_TARGETS = 5

# IMPORTANT: only mainland-India cells are eligible for OSM targeting.
baseline_india = baseline[baseline["in_mainland_india"]].copy()
recent_india = recent[recent["in_mainland_india"]].copy()

recurring = baseline_india[baseline_india["is_recurring"]].copy()

recurring_targets = (
    recurring[
        ["lat_grid", "lon_grid", "persistence_ratio", "detection_count", "frp_mean"]
    ]
    .drop_duplicates(["lat_grid", "lon_grid"])
    .sort_values(
        ["persistence_ratio", "detection_count", "frp_mean"],
        ascending=False
    )
    .head(MAX_RECURRING_TARGETS)
    [["lat_grid", "lon_grid"]]
    .copy()
)
recurring_targets["priority"] = 1
recurring_targets["reason"] = "recurring"

anomalies = recent_india[
    recent_india["frp_zscore"].notna() & (recent_india["frp_zscore"] >= 3)
].copy()

anomaly_targets = (
    anomalies[["lat_grid", "lon_grid", "frp_zscore", "frp"]]
    .drop_duplicates(["lat_grid", "lon_grid"])
    .sort_values(["frp_zscore", "frp"], ascending=False)
    .head(MAX_ANOMALY_TARGETS)
    [["lat_grid", "lon_grid"]]
    .copy()
)
anomaly_targets["priority"] = 2
anomaly_targets["reason"] = "recent_anomaly"

new_locations = recent_india[recent_india["has_baseline"] == False].copy()

new_targets = (
    new_locations[["lat_grid", "lon_grid", "frp"]]
    .drop_duplicates(["lat_grid", "lon_grid"])
    .sort_values("frp", ascending=False)
    .head(MAX_NEW_TARGETS)
    [["lat_grid", "lon_grid"]]
    .copy()
)
new_targets["priority"] = 3
new_targets["reason"] = "new_recent_location"

candidates = pd.concat(
    [recurring_targets, anomaly_targets, new_targets],
    ignore_index=True
)

candidates = (
    candidates
    .sort_values(["priority", "lat_grid", "lon_grid"])
    .drop_duplicates(["lat_grid", "lon_grid"], keep="first")
    .reset_index(drop=True)
)

candidates["target_id"] = np.arange(len(candidates))

print(f"Mainland recurring baseline cells available: {len(recurring):,}")
print(f"Mainland recent anomaly detections (z >= 3): {len(anomalies):,}")
print(f"Mainland recent new-location detections: {len(new_locations):,}")
print(f"Unique mainland-India OSM targets selected: {len(candidates)}")

display(candidates)

Mainland recurring baseline cells available: 289
Mainland recent anomaly detections (z >= 3): 51
Mainland recent new-location detections: 439
Unique mainland-India OSM targets selected: 30


,lat_grid,lon_grid,priority,reason,target_id
0,8.74,77.60,1,recurring,0
1,15.17,76.38,1,recurring,1
2,22.32,82.56,1,recurring,2
3,23.99,79.39,1,recurring,3
4,24.14,82.71,1,recurring,4
5,24.20,82.71,1,recurring,5
6,25.39,68.58,1,recurring,6
7,26.26,74.10,1,recurring,7
8,26.64,79.48,1,recurring,8
9,29.46,76.87,1,recurring,9


## 3. Conservative Overpass configuration

The previous main endpoint returned `429 Too Many Requests` and `504 Gateway Timeout`.

This version uses the Private.coffee Overpass instance as the primary endpoint and keeps the workload small. Public Overpass documentation lists it as a global public instance with no rate limit in place, while still asking users to share resources fairly.

**Do not increase these limits for the SIH MVP unless necessary.**

In [6]:
OSM_RADIUS_METERS = 1500
BATCH_SIZE = 3
REQUEST_DELAY_SECONDS = 4.0
REQUEST_TIMEOUT_SECONDS = 60

PRIMARY_OVERPASS_URL = "https://overpass.private.coffee/api/interpreter"
FALLBACK_OVERPASS_URL = "https://maps.mail.ru/osm/tools/overpass/api/interpreter"

USER_AGENT = (
    "GeoFlare-SIH2026/1.0 "
    "(educational disaster-management project; OSM context research)"
)

print("Primary:", PRIMARY_OVERPASS_URL)
print("Fallback:", FALLBACK_OVERPASS_URL)
print("Targets:", len(candidates))
print("Estimated requests:", math.ceil(len(candidates) / BATCH_SIZE))

Primary: https://overpass.private.coffee/api/interpreter
Fallback: https://maps.mail.ru/osm/tools/overpass/api/interpreter
Targets: 30
Estimated requests: 10


In [7]:
OSM_TAG_FILTERS = {
    "landuse": [
        'nwr["landuse"~"^(industrial|farmland|farmyard|orchard|vineyard|forest|residential|commercial)$"]'
    ],
    "industrial_building": [
        'nwr["building"="industrial"]'
    ],
    "works": [
        'nwr["man_made"="works"]'
    ],
    "wood": [
        'nwr["natural"="wood"]'
    ],
}

print("OSM query groups:", list(OSM_TAG_FILTERS))

OSM query groups: ['landuse', 'industrial_building', 'works', 'wood']


In [8]:
def build_overpass_query(target_rows, radius_m=OSM_RADIUS_METERS):
    clauses = []

    for _, row in target_rows.iterrows():
        lat = float(row["lat_grid"])
        lon = float(row["lon_grid"])

        for filters in OSM_TAG_FILTERS.values():
            for tag_filter in filters:
                clauses.append(
                    f'{tag_filter}(around:{radius_m},{lat:.5f},{lon:.5f});'
                )

    query = (
        "[out:json][timeout:55];\n"
        "(\n"
        + "  " + "\n  ".join(clauses)
        + "\n);\n"
        "out center;"
    )
    return query


def query_hash(query):
    return hashlib.sha1(query.encode("utf-8")).hexdigest()[:12]

print("Compact query builder ready.")

Compact query builder ready.


In [9]:
def cache_path_for_query(query):
    return CACHE_DIR / f"osm_{query_hash(query)}.json"


def load_cached_response(query):
    path = cache_path_for_query(query)
    if not path.exists():
        return None

    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def save_cached_response(query, payload):
    path = cache_path_for_query(query)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f)
    return path

print("Cache helpers ready.")

Cache helpers ready.


# 7A. OSM smoke test — RUN THIS FIRST

This cell makes **exactly one small request** using the first three
**mainland-India** targets.

### Stop here after running it.

Expected result:

```text
status: downloaded
```

or

```text
status: cached
```

The important checks are:

- target coordinates should look like Indian locations
- HTTP status should be `200`
- `elements` should ideally be greater than `0`

If you get `429`, `5xx`, timeout, or another error, **do not repeatedly rerun it**. Send me the output.

In [10]:
test_targets = candidates.head(BATCH_SIZE).copy()

if test_targets.empty:
    raise ValueError(
        "No mainland-India OSM targets were selected. "
        "Inspect the target-selection output before continuing."
    )

# Safety check: the smoke test must never query an excluded cell.
assert all(
    in_mainland_india(float(r.lat_grid), float(r.lon_grid))
    for r in test_targets.itertuples()
), "A non-mainland target slipped into the smoke test."

test_query = build_overpass_query(test_targets)
test_cache = load_cached_response(test_query)

print("Smoke-test targets:")
display(test_targets)

if test_cache is not None:
    test_payload = test_cache
    print(
        "Smoke-test status: cached | "
        f"elements={len(test_payload.get('elements', []))}"
    )
else:
    started = time.time()
    response = requests.post(
        PRIMARY_OVERPASS_URL,
        data={"data": test_query},
        headers={"User-Agent": USER_AGENT},
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    elapsed = time.time() - started

    print(f"Primary status code: {response.status_code}")
    print(f"Elapsed: {elapsed:.1f}s")

    if response.status_code == 200:
        test_payload = response.json()
        save_cached_response(test_query, test_payload)
        print(
            "Smoke-test status: downloaded | "
            f"elements={len(test_payload.get('elements', []))}"
        )
    else:
        print(response.text[:1000])
        raise RuntimeError(
            f"Smoke test failed with HTTP {response.status_code}. "
            "Do not rerun repeatedly; inspect the output first."
        )

Smoke-test targets:


,lat_grid,lon_grid,priority,reason,target_id
0,8.74,77.60,1,recurring,0
1,15.17,76.38,1,recurring,1
2,22.32,82.56,1,recurring,2


Primary status code: 200
Elapsed: 30.5s
Smoke-test status: downloaded | elements=24


# 7B. Full OSM collection — RUN ONLY AFTER 7A SUCCEEDS

This runs the small batches sequentially.

- successful responses are cached
- a 429 gets a real cooldown before one retry
- a 5xx/timeout can try the fallback endpoint once
- failed batches are recorded as `query_failed`
- the notebook continues rather than hanging indefinitely

In [11]:
# Final safety guard before full collection.
if not candidates.empty:
    assert all(
        in_mainland_india(float(r.lat_grid), float(r.lon_grid))
        for r in candidates.itertuples()
    ), "Non-mainland target detected. Stop and inspect candidates."

def request_overpass(query):
    cached = load_cached_response(query)

    if cached is not None:
        return {
            "status": "cached",
            "payload": cached,
            "endpoint": "cache",
            "error": None,
        }

    headers = {"User-Agent": USER_AGENT}

    for endpoint_name, endpoint_url in [
        ("primary", PRIMARY_OVERPASS_URL),
        ("fallback", FALLBACK_OVERPASS_URL),
    ]:
        try:
            started = time.time()

            response = requests.post(
                endpoint_url,
                data={"data": query},
                headers=headers,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )

            if response.status_code == 200:
                payload = response.json()
                save_cached_response(query, payload)

                return {
                    "status": "downloaded",
                    "payload": payload,
                    "endpoint": endpoint_name,
                    "error": None,
                    "elapsed": time.time() - started,
                }

            if response.status_code == 429:
                print(
                    f"{endpoint_name}: HTTP 429. "
                    "Waiting 30 seconds before one retry..."
                )
                time.sleep(30)

                retry = requests.post(
                    endpoint_url,
                    data={"data": query},
                    headers=headers,
                    timeout=REQUEST_TIMEOUT_SECONDS,
                )

                if retry.status_code == 200:
                    payload = retry.json()
                    save_cached_response(query, payload)

                    return {
                        "status": "downloaded_after_429",
                        "payload": payload,
                        "endpoint": endpoint_name,
                        "error": None,
                        "elapsed": time.time() - started,
                    }

                print(
                    f"{endpoint_name}: retry after 429 returned "
                    f"HTTP {retry.status_code}"
                )
                continue

            print(
                f"{endpoint_name}: HTTP {response.status_code}. "
                "Trying the next endpoint if available."
            )

        except requests.RequestException as exc:
            print(f"{endpoint_name}: request error: {exc}")

    return {
        "status": "query_failed",
        "payload": None,
        "endpoint": None,
        "error": "All Overpass endpoints failed",
    }


batch_records = []
batch_payloads = []

num_batches = math.ceil(len(candidates) / BATCH_SIZE)

for batch_number, start in enumerate(
    range(0, len(candidates), BATCH_SIZE), start=1
):
    batch_targets = candidates.iloc[start:start + BATCH_SIZE].copy()
    query = build_overpass_query(batch_targets)

    result = request_overpass(query)
    payload = result["payload"]

    element_count = (
        len(payload.get("elements", []))
        if payload is not None else 0
    )

    batch_records.append({
        "batch_number": batch_number,
        "targets": len(batch_targets),
        "status": result["status"],
        "endpoint": result["endpoint"],
        "elements": element_count,
        "target_ids": ",".join(
            batch_targets["target_id"].astype(str).tolist()
        ),
    })

    if payload is not None:
        batch_payloads.append({
            "batch_number": batch_number,
            "target_ids": batch_targets["target_id"].tolist(),
            "payload": payload,
        })

    print(
        f"Batch {batch_number}/{num_batches} | "
        f"targets={len(batch_targets)} | "
        f"status={result['status']} | "
        f"elements={element_count}"
    )

    if batch_number < num_batches:
        time.sleep(REQUEST_DELAY_SECONDS)

batch_log = pd.DataFrame(batch_records)

print("\nBatch summary:")
display(batch_log)

failed_batches = batch_log[batch_log["status"] == "query_failed"]
print(f"Failed batches: {len(failed_batches)}")

Batch 1/10 | targets=3 | status=cached | elements=24
primary: request error: HTTPSConnectionPool(host='overpass.private.coffee', port=443): Read timed out. (read timeout=60)
Batch 2/10 | targets=3 | status=downloaded | elements=58
primary: request error: HTTPSConnectionPool(host='overpass.private.coffee', port=443): Read timed out. (read timeout=60)
Batch 3/10 | targets=3 | status=downloaded | elements=7
Batch 4/10 | targets=3 | status=downloaded | elements=7
Batch 5/10 | targets=3 | status=downloaded | elements=3
primary: request error: HTTPSConnectionPool(host='overpass.private.coffee', port=443): Read timed out. (read timeout=60)
Batch 6/10 | targets=3 | status=downloaded | elements=10
primary: request error: HTTPSConnectionPool(host='overpass.private.coffee', port=443): Read timed out. (read timeout=60)
Batch 7/10 | targets=3 | status=downloaded | elements=272
Batch 8/10 | targets=3 | status=downloaded | elements=21
primary: request error: HTTPSConnectionPool(host='overpass.private

,batch_number,targets,status,endpoint,elements,target_ids
0,1,3,cached,cache,24,"0,1,2"
1,2,3,downloaded,fallback,58,"3,4,5"
2,3,3,downloaded,fallback,7,"6,7,8"
3,4,3,downloaded,primary,7,"9,10,11"
4,5,3,downloaded,primary,3,"12,13,14"
5,6,3,downloaded,fallback,10,"15,16,17"
6,7,3,downloaded,fallback,272,"18,19,20"
7,8,3,downloaded,primary,21,"21,22,23"
8,9,3,downloaded,fallback,3,"24,25,26"
9,10,3,downloaded,primary,12,"27,28,29"


Failed batches: 0


In [12]:
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2)
        * math.sin(dlambda / 2) ** 2
    )
    return 2 * R * math.asin(math.sqrt(a))


def element_center(element):
    center = element.get("center")
    if center and "lat" in center and "lon" in center:
        return float(center["lat"]), float(center["lon"])

    if element.get("lat") is not None and element.get("lon") is not None:
        return float(element["lat"]), float(element["lon"])

    return None


def element_categories(element):
    tags = element.get("tags", {})
    landuse = tags.get("landuse")
    natural = tags.get("natural")
    building = tags.get("building")
    man_made = tags.get("man_made")

    categories = set()

    if (
        landuse == "industrial"
        or building == "industrial"
        or man_made == "works"
    ):
        categories.add("industrial")

    if landuse in {"farmland", "farmyard", "orchard", "vineyard"}:
        categories.add("agriculture")

    if landuse == "forest" or natural == "wood":
        categories.add("forest")

    if landuse == "residential":
        categories.add("residential")

    if landuse == "commercial":
        categories.add("commercial")

    return categories


feature_rows = []

target_status = {}
for record in batch_records:
    status = (
        "ok"
        if record["status"] in {
            "downloaded",
            "downloaded_after_429",
            "cached",
        }
        else "query_failed"
    )
    for target_id in record["target_ids"].split(","):
        if target_id:
            target_status[int(target_id)] = status

for _, target in candidates.iterrows():
    target_id = int(target["target_id"])
    lat = float(target["lat_grid"])
    lon = float(target["lon_grid"])

    distances = {
        "industrial": [],
        "agriculture": [],
        "forest": [],
        "residential": [],
        "commercial": [],
    }

    for batch in batch_payloads:
        if target_id not in batch["target_ids"]:
            continue

        for element in batch["payload"].get("elements", []):
            center = element_center(element)
            if center is None:
                continue

            categories = element_categories(element)
            if not categories:
                continue

            distance = haversine_m(
                lat, lon, center[0], center[1]
            )

            if distance <= OSM_RADIUS_METERS:
                for category in categories:
                    distances[category].append(distance)

    row = {
        "target_id": target_id,
        "lat_grid": lat,
        "lon_grid": lon,
        "target_reason": target["reason"],
        "osm_query_status": target_status.get(
            target_id, "query_failed"
        ),
    }

    for category, values in distances.items():
        row[f"near_{category}"] = bool(values)
        row[f"nearest_{category}_m"] = (
            round(min(values), 1) if values else np.nan
        )

    feature_rows.append(row)

context_features = pd.DataFrame(feature_rows)

print(f"OSM context rows: {len(context_features)}")
display(context_features.head(10))

OSM context rows: 30


,target_id,lat_grid,lon_grid,target_reason,osm_query_status,near_industrial,nearest_industrial_m,near_agriculture,nearest_agriculture_m,near_forest,nearest_forest_m,near_residential,nearest_residential_m,near_commercial,nearest_commercial_m
0,0,8.74,77.60,recurring,ok,False,NaN,True,577.5,True,1290.6,False,NaN,False,NaN
1,1,15.17,76.38,recurring,ok,True,252.1,False,NaN,False,NaN,False,NaN,False,NaN
2,2,22.32,82.56,recurring,ok,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN
3,3,23.99,79.39,recurring,ok,True,65.6,False,NaN,False,NaN,True,1005.4,False,NaN
4,4,24.14,82.71,recurring,ok,True,211.6,False,NaN,False,NaN,False,NaN,False,NaN
5,5,24.20,82.71,recurring,ok,True,234.5,False,NaN,False,NaN,True,730.0,False,NaN
6,6,25.39,68.58,recurring,ok,True,625.3,False,NaN,False,NaN,True,294.2,False,NaN
7,7,26.26,74.10,recurring,ok,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN
8,8,26.64,79.48,recurring,ok,True,1151.9,False,NaN,False,NaN,False,NaN,False,NaN
9,9,29.46,76.87,recurring,ok,True,842.9,False,NaN,False,NaN,False,NaN,False,NaN


In [14]:
osm_join_cols = [
    "lat_grid", "lon_grid",
    "target_reason", "osm_query_status",
    "near_industrial", "nearest_industrial_m",
    "near_agriculture", "nearest_agriculture_m",
    "near_forest", "nearest_forest_m",
    "near_residential", "nearest_residential_m",
    "near_commercial", "nearest_commercial_m",
]

baseline_osm = baseline.merge(
    context_features[osm_join_cols],
    on=["lat_grid", "lon_grid"],
    how="left",
)

recent_osm = recent.merge(
    context_features[osm_join_cols],
    on=["lat_grid", "lon_grid"],
    how="left",
)

baseline_osm["osm_query_status"] = (
    baseline_osm["osm_query_status"].fillna("not_queried")
)
recent_osm["osm_query_status"] = (
    recent_osm["osm_query_status"].fillna("not_queried")
)

for df in [baseline_osm, recent_osm]:
    for category in [
        "industrial", "agriculture", "forest",
        "residential", "commercial"
    ]:
        df[f"near_{category}"] = (
            df[f"near_{category}"].fillna(False).astype(bool)
        )

print(
    f"Baseline rows preserved: {len(baseline_osm):,} / {len(baseline):,}"
)
print(
    f"Recent rows preserved: {len(recent_osm):,} / {len(recent):,}"
)

Baseline rows preserved: 4,855 / 4,855
Recent rows preserved: 1,112 / 1,112


In [15]:
ok_context = baseline_osm[
    (baseline_osm["is_recurring"]) &
    (baseline_osm["osm_query_status"] == "ok")
].copy()

summary = {
    "recurring_cells_with_successful_osm": len(ok_context),
    "industrial_context": int(ok_context["near_industrial"].sum()),
    "agriculture_context": int(ok_context["near_agriculture"].sum()),
    "forest_context": int(ok_context["near_forest"].sum()),
    "residential_context": int(ok_context["near_residential"].sum()),
    "commercial_context": int(ok_context["near_commercial"].sum()),
}

display(pd.DataFrame([summary]))

,recurring_cells_with_successful_osm,industrial_context,agriculture_context,forest_context,residential_context,commercial_context
0,19,13,2,1,5,0


In [16]:
candidate_view = (
    baseline_osm[
        (baseline_osm["is_recurring"]) &
        (baseline_osm["near_industrial"]) &
        (baseline_osm["osm_query_status"] == "ok")
    ]
    .sort_values(
        ["persistence_ratio", "detection_count", "frp_mean"],
        ascending=False
    )
    .head(25)
)

candidate_cols = [
    c for c in [
        "lat_grid", "lon_grid",
        "detection_count", "active_days",
        "persistence_ratio", "frp_mean", "frp_max",
        "nearest_industrial_m", "near_agriculture",
        "near_forest", "near_residential",
        "near_commercial", "osm_query_status"
    ]
    if c in candidate_view.columns
]

display(candidate_view[candidate_cols])

,lat_grid,lon_grid,detection_count,active_days,persistence_ratio,frp_mean,frp_max,nearest_industrial_m,near_agriculture,near_forest,near_residential,near_commercial,osm_query_status
3589,24.20,82.71,5,4,0.571,2.656000,5.51,234.5,False,False,True,False,ok
3689,26.64,79.48,17,11,0.500,1.700000,4.01,1151.9,False,False,False,False,ok
3894,29.46,76.87,7,4,0.500,7.044286,14.92,842.9,False,False,False,False,ok
4766,33.30,71.19,11,11,0.458,1.343636,3.45,315.2,False,False,False,False,ok
3633,25.39,68.58,32,26,0.448,1.701563,3.33,625.3,False,False,True,False,ok
2885,15.17,76.38,10,7,0.438,1.606000,5.97,252.1,False,False,False,False,ok
3572,24.14,82.71,4,3,0.429,1.305000,2.52,211.6,False,False,False,False,ok
4722,32.57,72.56,3,3,0.429,3.756667,4.10,572.0,False,False,True,False,ok
3555,23.99,79.39,3,3,0.429,1.686667,3.32,65.6,False,False,True,False,ok
2871,15.07,76.87,3,3,0.333,1.563333,1.61,233.7,False,False,False,False,ok


## 12. Save Notebook 03 outputs

These files are the handoff to Notebook 04.

In [17]:
baseline_osm_file = PROCESSED_DIR / "firms_baseline_osm.csv"
recent_osm_file = PROCESSED_DIR / "firms_recent_scored_osm.csv"
osm_context_file = PROCESSED_DIR / "osm_context_features.csv"
batch_log_file = PROCESSED_DIR / "osm_batch_log.csv"

baseline_osm.to_csv(baseline_osm_file, index=False)
recent_osm.to_csv(recent_osm_file, index=False)
context_features.to_csv(osm_context_file, index=False)
batch_log.to_csv(batch_log_file, index=False)

print(f"Saved: {baseline_osm_file} ({len(baseline_osm):,} rows)")
print(f"Saved: {recent_osm_file} ({len(recent_osm):,} rows)")
print(f"Saved: {osm_context_file} ({len(context_features):,} queried cells)")
print(f"Saved: {batch_log_file} ({len(batch_log):,} batches)")

Saved: c:\Users\ASUS\Documents\Sidra\ThermoTech\data\processed\firms_baseline_osm.csv (4,855 rows)
Saved: c:\Users\ASUS\Documents\Sidra\ThermoTech\data\processed\firms_recent_scored_osm.csv (1,112 rows)
Saved: c:\Users\ASUS\Documents\Sidra\ThermoTech\data\processed\osm_context_features.csv (30 queried cells)
Saved: c:\Users\ASUS\Documents\Sidra\ThermoTech\data\processed\osm_batch_log.csv (10 batches)


# 13. Phase 3 checkpoint ✅

Before moving to Notebook 04, confirm:

- [ ] Notebook 02 baseline and recent-scored files loaded
- [ ] Mainland-India geographic mask applied before OSM target selection
- [ ] Smoke-test coordinates are in India
- [ ] Smoke test returned HTTP 200
- [ ] Smoke test returned useful OSM elements
- [ ] Full Overpass collection completed or failed batches are explicitly recorded
- [ ] Successful responses were cached
- [ ] Failed requests were not interpreted as `False`
- [ ] Industrial/agriculture/forest/residential/commercial context features computed
- [ ] Nearest-distance features created where context was found
- [ ] Full baseline and recent row counts preserved
- [ ] Enriched datasets saved

### Outputs

```text
data/processed/
├── firms_baseline_osm.csv
├── firms_recent_scored_osm.csv
├── osm_context_features.csv
└── osm_batch_log.csv
```

### Next

`04_feature_engineering.ipynb`